### 策略名称: 15分钟价格&反转复合因子 (15min Intraday Composite: Low Price & Reversal)

**策略概述:**
本策略基于高频快照数据（Snapshot），构建了一个 15 分钟频率的复合因子。旨在捕捉日内交易中的两个主要特征：
1.  **截面低价效应**: 在同一时间窗口内，价格相对较低的股票未来预期收益更高（类似“捡便宜”逻辑）。
2.  **日内均值回归**: 短期内价格偏离均价过大时，倾向于向均值回归（反转逻辑）。
最终因子由上述两个子因子加权合成，输出标准化后的 [0, 1] 之间的信号值。

---

**数学逻辑 (Mathematical Logic):**

1.  **子因子 1: 截面低价因子 (Cross-sectional Low Price Factor)**
    * 对每个 15 分钟窗口内的标的按收盘中间价 ($P_{close}$) 从高到低排序。
    * $Rank_{desc}$: 价格最高的排名为 1，最低的排名为 $N$。
    * 标准化公式:
        $$ F_{price} = 1.0 - \frac{N - Rank_{desc}}{N - 1} $$
    * **逻辑**: 价格越低，排名越靠后 ($Rank \to N$)，因子值越接近 1；价格越高，因子值越接近 0。

2.  **子因子 2: 日内动量反转因子 (Intraday Momentum Reversal)**
    * 计算窗口内的收盘价 ($P_{close}$) 相对于均价 ($P_{avg}$) 的偏离度。
    * 使用双曲正切函数 ($\tanh$) 将偏离度映射到 $[-1, 1]$，并取负号表示反转：
        $$ F_{mom\_raw} = -1 \times \tanh\left( \frac{P_{close} - P_{avg}}{P_{avg}} \times 10 \right) $$
    * 归一化到 $[0, 1]$ 区间:
        $$ F_{mom} = \frac{F_{mom\_raw} + 1}{2} $$
    * **逻辑**: 价格高于均价（上涨）时，因子得分低（做空信号）；价格低于均价（下跌）时，因子得分高（做多信号）。

3.  **复合因子 (Composite Factor)**
    $$ F_{final} = w_{price} \times F_{price} + w_{mom} \times F_{mom} $$
    * 默认权重: $w_{price}=0.2, w_{mom}=0.8$。

---

**Args:**
* `datasource` (str): 数据源表名 (e.g., `'cpt_dwc_2026_stock_hs300_snapshot'`)
* `start_date` (str): 开始日期 `'YYYY-MM-DD HH:MM:SS'`
* `end_date` (str): 结束日期 `'YYYY-MM-DD HH:MM:SS'`

**Returns:**
* `pd.DataFrame`: 因子数据，包含 columns `['date', 'instrument', 'factor']`
    * 其中 `date` 为每个 15 分钟窗口的结束时间点。


In [ ]:
def main(datasource, start_date, end_date):
    """
    factor function
    构建 15分钟频率的复合因子 (Price + Intraday Momentum)

    组合方式：
    1) 低价因子（截面排序，低价=1，高价=0）
    2) 日内动量反转因子（窗口内 close vs avg，反转后映射到[0,1]）
    3) 复合：factor = w_price * price_factor + w_mom * mom_factor_01

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    # 你可以调权重（建议先 0.5/0.5）
    w_price = 0.2
    w_mom = 0.8

    sql = f"""
    -- 优化设置
    SET preserve_insertion_order=false;
    SET threads=4;

    WITH cte_snapshot AS (
        SELECT
            date,
            instrument_id,
            (ask_price1 + bid_price1) / 2.0 AS mid_price,
            strftime(date, '%Y-%m-%d') AS trading_day,

            CASE
                -- 上午
                WHEN strftime(date, '%H%M') >= '0930' AND strftime(date, '%H%M') < '0945' THEN 94500
                WHEN strftime(date, '%H%M') >= '0945' AND strftime(date, '%H%M') < '1000' THEN 100000
                WHEN strftime(date, '%H%M') >= '1000' AND strftime(date, '%H%M') < '1015' THEN 101500
                WHEN strftime(date, '%H%M') >= '1015' AND strftime(date, '%H%M') < '1030' THEN 103000
                WHEN strftime(date, '%H%M') >= '1030' AND strftime(date, '%H%M') < '1045' THEN 104500
                WHEN strftime(date, '%H%M') >= '1045' AND strftime(date, '%H%M') < '1100' THEN 110000
                WHEN strftime(date, '%H%M') >= '1100' AND strftime(date, '%H%M') < '1115' THEN 111500
                WHEN strftime(date, '%H%M') >= '1115' AND strftime(date, '%H%M') <= '1130' THEN 113000

                -- 下午
                WHEN strftime(date, '%H%M') >= '1300' AND strftime(date, '%H%M') < '1315' THEN 131500
                WHEN strftime(date, '%H%M') >= '1315' AND strftime(date, '%H%M') < '1330' THEN 133000
                WHEN strftime(date, '%H%M') >= '1330' AND strftime(date, '%H%M') < '1345' THEN 134500
                WHEN strftime(date, '%H%M') >= '1345' AND strftime(date, '%H%M') < '1400' THEN 140000
                WHEN strftime(date, '%H%M') >= '1400' AND strftime(date, '%H%M') < '1415' THEN 141500
                WHEN strftime(date, '%H%M') >= '1415' AND strftime(date, '%H%M') < '1430' THEN 143000
                WHEN strftime(date, '%H%M') >= '1430' AND strftime(date, '%H%M') < '1445' THEN 144500
                WHEN strftime(date, '%H%M') >= '1445' AND strftime(date, '%H%M') < '1457' THEN 150000
                ELSE -1
            END AS time_segment
        FROM {datasource}
    ),

    cte_filtered AS (
        SELECT *
        FROM cte_snapshot
        WHERE time_segment != -1
          AND mid_price IS NOT NULL
          AND mid_price > 0
    ),

    -- 每个 15min 窗口聚合：用于两个子因子
    cte_window AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,
            argMin(mid_price, date) AS open_mid,
            argMax(mid_price, date) AS close_mid,
            avg(mid_price)         AS avg_mid
        FROM cte_filtered
        GROUP BY instrument_id, trading_day, time_segment
    ),

    -- 子因子1：价格排序（低价=1，高价=0）
    cte_price_rank AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,

            row_number() OVER (
                PARTITION BY trading_day, time_segment
                ORDER BY close_mid DESC, instrument_id
            ) AS rn_desc,

            count(*) OVER (
                PARTITION BY trading_day, time_segment
            ) AS cnt_cs
        FROM cte_window
    ),

    cte_price_factor AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,
            CASE
                WHEN cnt_cs <= 1 THEN 0.5
                ELSE 1.0 - (cnt_cs - rn_desc) * 1.0 / (cnt_cs - 1)
            END AS price_factor_01
        FROM cte_price_rank
    ),

    -- 子因子2：动量反转（映射到[0,1]）
    cte_mom_factor AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,

            -- 原定义：-tanh((close-avg)/avg * 10) ∈ [-1,1]
            -1.0 * tanh(((close_mid - avg_mid) / (avg_mid + 1e-8)) * 10.0) AS mom_factor_raw
        FROM cte_window
    ),

    cte_mom_factor_01 AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,
            (mom_factor_raw + 1.0) / 2.0 AS mom_factor_01
        FROM cte_mom_factor
    ),

    -- 组合
    cte_combo AS (
        SELECT
            p.instrument_id,
            p.trading_day,
            p.time_segment,

            p.price_factor_01,
            m.mom_factor_01,

            ({w_price} * p.price_factor_01 + {w_mom} * m.mom_factor_01) AS factor
        FROM cte_price_factor p
        INNER JOIN cte_mom_factor_01 m
            ON p.instrument_id = m.instrument_id
           AND p.trading_day   = m.trading_day
           AND p.time_segment  = m.time_segment
    )

    SELECT
        CAST(CONCAT(
            c.trading_day, ' ',
            strftime(strptime(LPAD(c.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        c.factor
    FROM cte_combo c
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [start_date, end_date]}).df()
    return df


if __name__ == "__main__":
    """
    开发调试专用模块：分块循环回测引擎
    """
    from bigmodule import M
    import pandas as pd
    import structlog
    import gc

    logger = structlog.get_logger()
    datasource = "cpt_dwc_2026_stock_hs300_snapshot"

    full_start_date = "2023-01-01"
    full_end_date = "2024-12-01"

    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq="MS")
    all_results = []

    logger.info(f"🚀 Starting Combo Factor Backtest: {full_start_date} to {full_end_date}")

    for start_dt in date_ranges:
        current_start = start_dt.strftime("%Y-%m-%d 00:00:00")
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d 23:59:59")
        logger.info(f"Processing Chunk: {current_start} => {current_end}")

        try:
            df_chunk = main(datasource, current_start, current_end)

            if df_chunk is not None and not df_chunk.empty:
                all_results.append(df_chunk)
                logger.info(f"✅ Chunk Done. Rows: {len(df_chunk)}")
            else:
                logger.warning(f"⚠️ Chunk Empty: {current_start}")

            del df_chunk
            gc.collect()

        except Exception as e:
            logger.error(f"❌ Error in chunk {current_start}: {e}")

    if all_results:
        logger.info("🧩 Concatenating all chunks...")
        final_data = pd.concat(all_results, ignore_index=True)
        final_data.sort_values(by=["date", "instrument"], inplace=True)

        logger.info(f"🎉 All Done! Final Shape: {final_data.shape}")
        logger.info(f"Sample:\n{final_data.head()}")

        logger.info("📊 Starting Evaluation...")
        try:
            _ = M.eval_dwc._latest(data=final_data)
        except Exception as e:
            logger.warning(f"Evaluation failed (local env might miss modules): {e}")
            print("Data preview:", final_data.head())
    else:
        logger.error("No data generated.")
